# Conditional register addition and subtraction

Copyright 2024 Google LLC. Adapted under Apache-2.0 from Qualtran v0.7.0, commit `bcb17bb25360ed68ce3bc084645db3e929799389`.


In [ ]:
from collections import Counter
from collections.abc import Callable, Hashable
from dataclasses import dataclass
from html import escape
from typing import Literal, cast

from IPython import display as notebook_display
from sympy import Symbol

from examples.algorithms.register_controlled_add_or_subtract import (
    main as execute_round_trip,
)
from flagquantum.compiler._component_trace.arithmetic_call import ArithmeticCall
from flagquantum.compiler._component_trace.call_graph import (
    SymbolicCallGraph,
    symbolic_call_graph,
)
from flagquantum.compiler._component_trace.resource_generalization import (
    CliffordCall,
    PredicateControlledCall,
)
from flagquantum.compiler.registers.arithmetic.addition import (
    measurement_add_or_subtract_ir,
)
from flagquantum.compiler.registers.resources.symbolic_resource_model import (
    source_controlled_add_or_subtract_resource_formula,
)
from flagquantum.core.ir.program.register_types import (
    signed_register_type,
    unsigned_register_type,
)
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.core.value_math import SymbolicInt
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.register_graph_table import signature_instance_table
from flagquantum.drawer.register_score import RegisterStep

# Optional IPython functions enter through the presentation boundary.
SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


@dataclass(frozen=True)
class ConditionalExample:
    width: SymbolicInt
    signed: bool
    add_on: bool = True

    @property
    def signature(self) -> Signature:
        dtype = signed_register_type if self.signed else unsigned_register_type
        return Signature(
            (
                Register("ctrl", unsigned_register_type(1)),
                Register("a", dtype(self.width)),
                Register("b", dtype(self.width)),
            )
        )

    def direct_calls(self) -> dict[Hashable, SymbolicInt]:
        encoding: Literal["twos_complement", "unsigned"] = (
            "twos_complement" if self.signed else "unsigned"
        )
        calls: dict[Hashable, SymbolicInt] = {
            PredicateControlledCall(
                ArithmeticCall("bitwise_not", (self.width,), (encoding,)), (1,)
            ): 2,
            ArithmeticCall("add", (self.width, self.width), (encoding, encoding)): 1,
        }
        if self.add_on:
            calls[CliffordCall("x", 1)] = 2
        return calls


def signature_svg(items: tuple[tuple[str, ConditionalExample], ...]) -> bytes:
    nodes = []
    for i, (name, example) in enumerate(items):
        signature = example.signature
        ports = tuple(r.name for r in signature)
        table = signature_instance_table(
            signature,
            RegisterStep(name, ports, ports),
            {r.name: f"p{j}" for j, r in enumerate(signature)},
        )
        metadata = ", ".join(
            f"{r.name}={r.dtype.encoding}[{r.bitsize}]" for r in signature
        )
        table = table.replace(
            "</TABLE>", f'<TR><TD COLSPAN="2">{escape(metadata)}</TD></TR></TABLE>'
        )
        nodes.append(f"n{i} [shape=plain,label=<{table}>];")
    return render_call_graph_svg("digraph signatures { " + " ".join(nodes) + " }")


def call_title(call: Hashable) -> str:
    if isinstance(call, ConditionalExample):
        return f"Conditional add/subtract ({call.width} bits)"
    if isinstance(call, PredicateControlledCall):
        return "Controlled register complement"
    if isinstance(call, ArithmeticCall):
        return f"Add ({call.widths[0]} bits)"
    if isinstance(call, CliffordCall):
        return call.opcode.upper()
    return str(call)


## Conditional addition/subtraction
Adds or subtracts in-place into the target, based on a control bit.

Applies the transformation

$$
    |1\rangle |a\rangle |b\rangle \mapsto |1\rangle |a\rangle |b + a\rangle \\
    |0\rangle |a\rangle |b\rangle \mapsto |0\rangle |a\rangle |b - a\rangle
$$

Given two numbers `a`, `b` and a control bit `ctrl`, this bloq computes:

- the sum `b + a` when `ctrl=1`,
- the difference `b - a` when `ctrl=0`,

and stores the result in the second register (`b`).

This uses an uncontrolled `Add` surrounded by controlled `BitwiseNot`s, and only
the `Add` requires T gates, which has half the T-cost of a controlled `Add`.


#### Parameters
 - `a_dtype`: dtype of the lhs `a`
 - `b_dtype`: dtype of the rhs `b`. If it is not big enough to store the result, the most significant bits are dropped on overflow.
 - `add_when_ctrl_is_on`: If True (default), add when `ctrl=1` and subtract when `ctrl=0`. If False, add when `ctrl=0` and subtract when `ctrl=1`. This follows the original body and adjoint, correcting its contradictory argument prose. 

#### Registers
 - `ctrl`: a single control bit
 - `a`: an integer value.
 - `b`: an integer value. 

#### References
 - [Compilation of Fault-Tolerant Quantum Heuristics for Combinatorial Optimization](https://arxiv.org/abs/2007.07391). Sanders et al. Section II-A-1, Algorithm 1.


In [ ]:
# Concrete and symbolic examples share native signature and resource owners.


### Example Instances

In [ ]:
ctrl_add_or_sub_unsigned = ConditionalExample(8, signed=False)


In [ ]:
ctrl_add_or_sub_signed = ConditionalExample(8, signed=True)


In [ ]:
n = Symbol("n")
ctrl_add_or_sub_signed_symb = ConditionalExample(n, signed=True)


#### Graphical Signature

In [ ]:
examples = (
    ("signed_symbolic", ctrl_add_or_sub_signed_symb),
    ("unsigned", ctrl_add_or_sub_unsigned),
    ("signed", ctrl_add_or_sub_signed),
)
display(SVG(signature_svg(examples)))


### Call Graph

In [ ]:
root: Hashable = ctrl_add_or_sub_signed_symb
conditional_graph: SymbolicCallGraph[Hashable] = symbolic_call_graph(
    root, {root: ctrl_add_or_sub_signed_symb.direct_calls()}, max_depth=1
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                conditional_graph.root, conditional_graph.direct_calls, title=call_title
            )
        )
    )
)
display(conditional_graph.leaf_counts)


## Native execution and separate resource accounts

The existing append example verifies coherent control, complete complex amplitudes, clean carries and restoration by the opposite polarity. The eight-bit inventory below is measured from emitted primitives. The independent symbolic model retains its own peak width and categories. All-small-branch source comparisons remain in the complete conditional arithmetic audit.


In [ ]:
execute_round_trip()
gates = measurement_add_or_subtract_ir(
    0,
    tuple(range(1, 9)),
    tuple(range(9, 17)),
    tuple(range(17, 24)),
    tuple(range(7)),
    add_when_ctrl_is_on=True,
)
measured_primitives = dict(Counter(g.name for g in gates))
modeled_resources = source_controlled_add_or_subtract_resource_formula(8, 8)
display(measured_primitives)
display(modeled_resources)
